# NQ Research Platform

Research NQ futures setups around the **9:30 AM New York open**, entirely in your browser.

### How to use this page
1. Click **Runtime → Run all** in the menu at the top. (Or press the ▶ button on each box, top to bottom.)
2. If Google shows **"Warning: This notebook was not authored by Google"**, click **Run anyway**.
3. The first time, Google asks **"Permit this notebook to access your Google Drive files?"**
   Click **Connect to Google Drive**, pick your Google account, and click **Continue** / **Allow**.
   This lets the notebook save your price history in a Drive folder called **NQ Research**.
4. Wait for each box to finish. Results appear underneath it.

To change a setting, edit it in the **Settings** box, then click **Runtime → Run all** again.

Each grey box has hidden code. Double-click a box's title to see the code. You don't need to.

In [ ]:
#@title ⚙️ Settings
#@markdown **Where the price data comes from**
DATA_SOURCE = "Yahoo (free, last 60 days)" #@param ["Yahoo (free, last 60 days)", "Upload a TradingView export"]

#@markdown **Keep a growing price history in Google Drive.** Yahoo only gives the last 60 days,
#@markdown so each run adds the new days to a saved file. Run this at least once a month and
#@markdown your history keeps growing. Only used with the Yahoo data source.
SAVE_HISTORY_TO_DRIVE = True #@param {type:"boolean"}

#@markdown **The open** (New York time). 09:30 = stock market cash open, 08:30 = economic data releases.
OPEN_TIME = "09:30" #@param ["09:30", "08:30"]

#@markdown **Opening range length**, in minutes after the open
OPENING_RANGE_MINUTES = 15 #@param {type:"integer"}

#@markdown **How long to keep watching price** after the open, in minutes
WATCH_MINUTES = 120 #@param {type:"integer"}

#@markdown **Day to draw on the chart** in step 5, written like 2026-10-06. Leave empty for the most recent trade.
CHART_DAY = "" #@param {type:"string"}

SYMBOL = "NQ=F"          # Yahoo's name for front-month NQ futures
TIMEZONE = "America/New_York"
print("Settings saved.")

In [ ]:
#@title 1️⃣ Load price data
import io

import pandas as pd

try:
    import yfinance as yf
except ImportError:
    # Colab normally has this already; install it if it's missing.
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf


def read_tradingview(file_bytes):
    """Turn a TradingView "Export chart data" CSV into our candle table."""
    raw = pd.read_csv(io.BytesIO(file_bytes))
    raw.columns = [column.strip().lower() for column in raw.columns]

    missing = [c for c in ["time", "open", "high", "low", "close"] if c not in raw.columns]
    if missing:
        raise SystemExit(f"This doesn't look like a TradingView export. Missing columns: {missing}")

    # TradingView exports time as "UNIX timestamp" (a big number) or "ISO time" (readable text).
    if pd.api.types.is_numeric_dtype(raw["time"]):
        times = pd.to_datetime(raw["time"], unit="s", utc=True)
    else:
        times = pd.to_datetime(raw["time"], utc=True)

    table = pd.DataFrame({
        "Open": raw["open"].values,
        "High": raw["high"].values,
        "Low": raw["low"].values,
        "Close": raw["close"].values,
        "Volume": raw["volume"].values if "volume" in raw.columns else 0,
    }, index=times)
    return table


if DATA_SOURCE.startswith("Yahoo"):
    print("Downloading 60 days of 5-minute NQ candles from Yahoo...")
    candles = yf.download(SYMBOL, period="60d", interval="5m", progress=False, auto_adjust=False)
    if candles.empty:
        raise SystemExit("No data came back from Yahoo. Wait a minute and run this box again.")
    # Yahoo gives two-level column names ("Close" / "NQ=F"). Keep the first level only.
    candles.columns = candles.columns.get_level_values(0)
    candles = candles[["Open", "High", "Low", "Close", "Volume"]]
else:
    from google.colab import files
    print("Click 'Choose Files' below and pick your TradingView CSV export.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was chosen. Run this box again.")
    file_name, file_bytes = next(iter(uploaded.items()))
    print(f"Reading {file_name}...")
    candles = read_tradingview(file_bytes)

candles.index = candles.index.tz_convert(TIMEZONE)
candles.index.name = "Time"
candles = candles.sort_index()

if DATA_SOURCE.startswith("Yahoo") and SAVE_HISTORY_TO_DRIVE:
    import os
    from google.colab import drive
    drive.mount("/content/drive")
    history_folder = "/content/drive/MyDrive/NQ Research"
    history_file = f"{history_folder}/nq_5m_history.csv"
    os.makedirs(history_folder, exist_ok=True)

    if os.path.exists(history_file):
        saved = pd.read_csv(history_file, index_col="Time")
        saved.index = pd.to_datetime(saved.index, utc=True).tz_convert(TIMEZONE)
        print(f"Found {len(saved):,} saved candles in Google Drive.")
        # Put the saved and new candles together. Where both have the same
        # candle, keep the new one (Yahoo sometimes corrects recent prices).
        candles = pd.concat([saved, candles])
        candles = candles[~candles.index.duplicated(keep="last")].sort_index()

    candles.to_csv(history_file)
    print(f"Saved {len(candles):,} candles to Google Drive: My Drive/NQ Research/nq_5m_history.csv")
    print()

minutes = candles.index.to_series().diff().dropna().dt.total_seconds().median() / 60
print(f"Loaded {len(candles):,} candles of about {minutes:g} minutes each")
print(f"From {candles.index[0]:%Y-%m-%d %H:%M} to {candles.index[-1]:%Y-%m-%d %H:%M} New York time")
print(f"Trading days: {len(set(candles.index.date))}")

In [ ]:
#@title 2️⃣ Opening range study
#@markdown For every day: the high and low of the opening range, which side broke first,
#@markdown and how far price travelled past the range afterwards.
from datetime import timedelta

from IPython.display import display

open_hour, open_minute = map(int, OPEN_TIME.split(":"))
results = []

for day, day_candles in candles.groupby(candles.index.date):
    open_time = pd.Timestamp(day, tz=TIMEZONE).replace(hour=open_hour, minute=open_minute)
    range_end = open_time + timedelta(minutes=OPENING_RANGE_MINUTES)
    watch_end = open_time + timedelta(minutes=WATCH_MINUTES)

    range_candles = day_candles[(day_candles.index >= open_time) & (day_candles.index < range_end)]
    after_candles = day_candles[(day_candles.index >= range_end) & (day_candles.index < watch_end)]

    # Skip weekends, holidays, and days with missing data.
    if range_candles.empty or after_candles.empty:
        continue

    range_high = range_candles["High"].max()
    range_low = range_candles["Low"].min()
    broke_up = after_candles[after_candles["High"] > range_high]
    broke_down = after_candles[after_candles["Low"] < range_low]

    if broke_up.empty and broke_down.empty:
        first_break = "none"
    elif broke_down.empty or (not broke_up.empty and broke_up.index[0] < broke_down.index[0]):
        first_break = "up"
    elif broke_up.empty or broke_down.index[0] < broke_up.index[0]:
        first_break = "down"
    else:
        # Both sides broke inside the same candle - we can't tell which came first.
        first_break = "same candle"

    results.append({
        "date": day,
        "range_high": range_high,
        "range_low": range_low,
        "range_size": range_high - range_low,
        "first_break": first_break,
        "broke_up": not broke_up.empty,
        "broke_down": not broke_down.empty,
        "max_points_above": max(after_candles["High"].max() - range_high, 0),
        "max_points_below": max(range_low - after_candles["Low"].min(), 0),
    })

if not results:
    raise SystemExit(f"No days had data at {OPEN_TIME}. Check the data loaded in step 1.")

table = pd.DataFrame(results)

print(f"Opening range: first {OPENING_RANGE_MINUTES} minutes after {OPEN_TIME} New York")
print(f"Watching price until {WATCH_MINUTES} minutes after the open")
print()
print("=========== SUMMARY ===========")
print(f"Days studied:                  {len(table)}")
print(f"Average range size:            {table['range_size'].mean():.1f} points")
print(f"Days that broke up first:      {(table['first_break'] == 'up').sum()}")
print(f"Days that broke down first:    {(table['first_break'] == 'down').sum()}")
print(f"Days that broke BOTH sides:    {(table['broke_up'] & table['broke_down']).sum()}")
print(f"Days that never broke out:     {(table['first_break'] == 'none').sum()}")
print(f"Average move above the range:  {table['max_points_above'].mean():.1f} points")
print(f"Average move below the range:  {table['max_points_below'].mean():.1f} points")
print()
display(table.round(2))

In [ ]:
#@title 3️⃣ Backtest: opening range breakout
#@markdown Simulates your trade on every day in the history: wait for the opening range
#@markdown to finish, then trade the **first** side that breaks. One trade per day.

#@markdown **How you enter**
#@markdown - *On the break*: a stop order 1 tick beyond the range (fills the moment price breaks it)
#@markdown - *On a candle close*: wait for a 5-minute candle to close outside the range, enter at that close
ENTRY = "On the break" #@param ["On the break", "On a candle close"]

#@markdown **Which direction**
DIRECTION = "Both" #@param ["Both", "Long only", "Short only"]

#@markdown **Where your stop goes**
STOP = "Other side of the range" #@param ["Other side of the range", "Middle of the range", "Fixed number of points"]
#@markdown Only used with *Fixed number of points*:
STOP_POINTS = 40 #@param {type:"number"}

#@markdown **Target**, as a multiple of your risk ("R"). 2 = aim to win twice what you risk.
#@markdown Set to 0 for no target (hold until the exit time).
TARGET_R = 2 #@param {type:"number"}

#@markdown **Exit any open trade at** (New York time)
EXIT_TIME = "11:30" #@param {type:"string"}

#@markdown **Skip days when the opening range is bigger than** this many points (0 = never skip)
MAX_RANGE_POINTS = 0 #@param {type:"number"}

#@markdown **Contract and costs**
CONTRACT = "NQ ($20 per point)" #@param ["NQ ($20 per point)", "MNQ ($2 per point)"]
CONTRACTS = 1 #@param {type:"integer"}
COMMISSION_PER_CONTRACT = 5 #@param {type:"number"}
#@markdown Slippage: how many ticks (0.25 points each) worse than planned your stop-order and market fills are
SLIPPAGE_TICKS = 1 #@param {type:"integer"}

from datetime import timedelta

from IPython.display import display

TICK = 0.25


def backtest_day(day, day_candles, settings):
    """Simulate one day. Returns (trade dictionary or None, reason it was skipped)."""
    s = settings
    tick_slip = s["slippage_ticks"] * TICK
    open_time = pd.Timestamp(day, tz=TIMEZONE).replace(hour=open_hour, minute=open_minute)
    range_end = open_time + timedelta(minutes=OPENING_RANGE_MINUTES)
    exit_hour, exit_minute = map(int, s["exit_time"].split(":"))
    exit_time = open_time.replace(hour=exit_hour, minute=exit_minute)

    range_candles = day_candles[(day_candles.index >= open_time) & (day_candles.index < range_end)]
    after = day_candles[(day_candles.index >= range_end) & (day_candles.index < exit_time)]
    if range_candles.empty or after.empty:
        return None, "no data"

    range_high = range_candles["High"].max()
    range_low = range_candles["Low"].min()
    range_size = range_high - range_low
    if s["max_range_points"] and range_size > s["max_range_points"]:
        return None, "range too big"

    # ---- 1. Find the entry ----
    side = None
    for position, (time, candle) in enumerate(after.iterrows()):
        if s["entry"] == "On the break":
            up = candle["High"] >= range_high + TICK
            down = candle["Low"] <= range_low - TICK
            if up and down:
                return None, "broke both sides in one candle"
            if up:
                side, entry = 1, max(range_high + TICK, candle["Open"]) + tick_slip
            elif down:
                side, entry = -1, min(range_low - TICK, candle["Open"]) - tick_slip
        else:
            if candle["Close"] > range_high:
                side, entry = 1, candle["Close"] + tick_slip
            elif candle["Close"] < range_low:
                side, entry = -1, candle["Close"] - tick_slip
        if side:
            break

    if side is None:
        return None, "no breakout"
    if (side == 1 and s["direction"] == "Short only") or (side == -1 and s["direction"] == "Long only"):
        return None, "broke the other direction"

    # ---- 2. Place the stop and target ----
    if s["stop"] == "Other side of the range":
        stop = range_low - TICK if side == 1 else range_high + TICK
    elif s["stop"] == "Middle of the range":
        stop = (range_high + range_low) / 2
    else:
        stop = entry - side * s["stop_points"]
    risk = (entry - stop) * side
    if risk <= 0:
        return None, "stop on the wrong side of entry"
    target = entry + side * s["target_r"] * risk if s["target_r"] > 0 else None

    # ---- 3. Walk forward candle by candle until the stop, target, or exit time ----
    # With "On the break", the entry candle itself can also hit the stop or target.
    # With "On a candle close", we entered at the close, so start from the next candle.
    first = position if s["entry"] == "On the break" else position + 1
    exit_price, exit_reason, exit_at = None, None, None
    for time, candle in after.iloc[first:].iterrows():
        low, high, open_ = candle["Low"], candle["High"], candle["Open"]
        stop_hit = (low <= stop) if side == 1 else (high >= stop)
        target_hit = target is not None and ((high >= target) if side == 1 else (low <= target))
        # If the stop and target are both inside one candle we can't know which came
        # first, so we assume the stop (the cautious choice).
        if stop_hit:
            gapped = (open_ < stop) if side == 1 else (open_ > stop)
            fill = open_ if (gapped and time != after.index[position]) else stop
            exit_price, exit_reason, exit_at = fill - side * tick_slip, "stop", time
            break
        if target_hit:
            exit_price, exit_reason, exit_at = target, "target", time
            break

    if exit_price is None:
        last_time, last = after.index[-1], after.iloc[-1]
        exit_price, exit_reason, exit_at = last["Close"] - side * tick_slip, "time", last_time

    points = (exit_price - entry) * side
    dollars = points * s["point_value"] * s["contracts"] - s["commission"] * s["contracts"]
    return {
        "date": day,
        "direction": "long" if side == 1 else "short",
        "entry_time": after.index[position].strftime("%H:%M"),
        "entry": entry,
        "stop": stop,
        "target": target,
        "exit_time": exit_at.strftime("%H:%M"),
        "exit": exit_price,
        "exit_reason": exit_reason,
        "range_size": range_size,
        "risk_points": risk,
        "points": points,
        "R": points / risk,
        "dollars": dollars,
    }, None


settings = {
    "entry": ENTRY, "direction": DIRECTION, "stop": STOP, "stop_points": STOP_POINTS,
    "target_r": TARGET_R, "exit_time": EXIT_TIME, "max_range_points": MAX_RANGE_POINTS,
    "point_value": 20 if CONTRACT.startswith("NQ") else 2, "contracts": CONTRACTS,
    "commission": COMMISSION_PER_CONTRACT, "slippage_ticks": SLIPPAGE_TICKS,
}

trades_list, skipped = [], []
for day, day_candles in candles.groupby(candles.index.date):
    trade, reason = backtest_day(day, day_candles, settings)
    if trade:
        trades_list.append(trade)
    elif reason != "no data":
        skipped.append(reason)

trades = pd.DataFrame(trades_list)
print(f"Backtest finished: {len(trades)} trades.")
if skipped:
    print("Days with no trade:", ", ".join(f"{n} {why}" for why, n in pd.Series(skipped).value_counts().items()))
if not trades.empty:
    display(trades.round(2))


In [ ]:
#@title 4️⃣ Score card
#@markdown How good the backtest was, in plain numbers, plus the account growth chart.
import plotly.graph_objects as go

if trades.empty:
    raise SystemExit("No trades to score. Try different settings in step 3.")

wins = trades[trades["dollars"] > 0]
losses = trades[trades["dollars"] <= 0]
equity = trades["dollars"].cumsum()
peak = equity.cummax().clip(lower=0)          # the account starts at $0, so that's the first peak
max_drawdown = (equity - peak).min()
profit_factor = wins["dollars"].sum() / abs(losses["dollars"].sum()) if not losses.empty else float("inf")

losing_streak = longest_losing_streak = 0
for dollars in trades["dollars"]:
    losing_streak = losing_streak + 1 if dollars <= 0 else 0
    longest_losing_streak = max(longest_losing_streak, losing_streak)

print("=========== SCORE CARD ===========")
print(f"Trades:                    {len(trades)}")
print(f"Win rate:                  {len(wins) / len(trades):.0%}")
print(f"Average winner:            ${wins['dollars'].mean() if not wins.empty else 0:,.0f}")
print(f"Average loser:             ${losses['dollars'].mean() if not losses.empty else 0:,.0f}")
print(f"Profit factor:             {profit_factor:.2f}   (money won / money lost; above 1 = profitable)")
print(f"Average result per trade:  ${trades['dollars'].mean():,.0f}   ({trades['R'].mean():+.2f} R)")
print(f"Total profit:              ${trades['dollars'].sum():,.0f}")
print(f"Worst drawdown:            ${max_drawdown:,.0f}   (biggest drop from a high point)")
print(f"Longest losing streak:     {longest_losing_streak} trades in a row")
print()
print("How trades ended:", ", ".join(f"{n} {why}" for why, n in trades["exit_reason"].value_counts().items()))
print()

by_direction = trades.groupby("direction").agg(
    trades=("dollars", "size"),
    win_rate=("dollars", lambda d: f"{(d > 0).mean():.0%}"),
    average_R=("R", "mean"),
    total_dollars=("dollars", "sum"),
)
display(by_direction.round(2))

if len(trades) < 30:
    print(f"⚠️ Only {len(trades)} trades. That's too few to trust - luck can easily explain")
    print("   results this small. Keep running the notebook so your history grows.")

figure = go.Figure(go.Scatter(
    x=pd.to_datetime(trades["date"]), y=equity, mode="lines+markers",
    line=dict(color="#2a78d6", width=2), marker=dict(size=8, color="#2a78d6"),
    hovertemplate="%{x|%b %d}<br>Account: $%{y:,.0f}<extra></extra>",
))
figure.add_hline(y=0, line_color="#9b9a95", line_width=1)
figure.update_layout(
    title=f"Account growth ({CONTRACTS} {CONTRACT.split()[0]}, after costs)",
    yaxis_title="Profit ($)", yaxis_tickprefix="$", height=420, template="plotly_white",
)
figure.show()

In [ ]:
#@title 5️⃣ Chart one day
#@markdown Candles from 1 hour before the open to the exit time. Grey lines = opening range.
#@markdown If a trade happened: blue triangle = entry, black X = exit, dashed red = stop, dashed green = target.
#@markdown Pick the day with **CHART_DAY** in the Settings box (empty = most recent trade).
import plotly.graph_objects as go

if CHART_DAY.strip():
    chart_day = pd.Timestamp(CHART_DAY).date()
elif not trades.empty:
    chart_day = trades["date"].iloc[-1]
else:
    chart_day = table["date"].iloc[-1]

day_row = table[table["date"] == chart_day]
if day_row.empty:
    raise SystemExit(f"{chart_day} isn't in the study. Pick a date from the table in step 2.")
day_row = day_row.iloc[0]
day_trade = trades[trades["date"] == chart_day].iloc[0] if not trades.empty and (trades["date"] == chart_day).any() else None

open_time = pd.Timestamp(chart_day, tz=TIMEZONE).replace(hour=open_hour, minute=open_minute)
exit_hour, exit_minute = map(int, EXIT_TIME.split(":"))
window = candles[(candles.index >= open_time - timedelta(hours=1))
                 & (candles.index <= open_time.replace(hour=exit_hour, minute=exit_minute))]

figure = go.Figure(go.Candlestick(
    x=window.index, open=window["Open"], high=window["High"],
    low=window["Low"], close=window["Close"], name="NQ",
))
figure.add_hline(y=day_row["range_high"], line_color="#52514e", annotation_text="Range high")
figure.add_hline(y=day_row["range_low"], line_color="#52514e", annotation_text="Range low", annotation_position="bottom right")
figure.add_vline(x=open_time, line_dash="dot", line_color="#9b9a95")

title = f"NQ {chart_day}  -  no trade"
if day_trade is not None:
    def at(clock):
        return pd.Timestamp(f"{chart_day} {clock}", tz=TIMEZONE)
    figure.add_hline(y=day_trade["stop"], line_dash="dash", line_color="#d03b3b", annotation_text="Stop", annotation_position="top left")
    if pd.notna(day_trade["target"]):
        figure.add_hline(y=day_trade["target"], line_dash="dash", line_color="#008300", annotation_text="Target", annotation_position="top left")
    figure.add_trace(go.Scatter(
        x=[at(day_trade["entry_time"])], y=[day_trade["entry"]], mode="markers", name="Entry",
        marker=dict(size=14, color="#2a78d6", symbol="triangle-up" if day_trade["direction"] == "long" else "triangle-down"),
    ))
    figure.add_trace(go.Scatter(
        x=[at(day_trade["exit_time"])], y=[day_trade["exit"]], mode="markers", name="Exit",
        marker=dict(size=12, color="#0b0b0b", symbol="x"),
    ))
    title = (f"NQ {chart_day}  -  {day_trade['direction']}, exit by {day_trade['exit_reason']}, "
             f"{day_trade['R']:+.2f} R (${day_trade['dollars']:,.0f})")

figure.update_layout(title=title, xaxis_rangeslider_visible=False, height=550, template="plotly_white")
figure.show()

In [ ]:
#@title 💾 Download the results
#@markdown Saves two files you can open in Excel, Numbers or Google Sheets:
#@markdown the opening range study and the list of backtest trades.
#@markdown If your browser asks to allow multiple downloads, click **Allow**.
table.to_csv("opening_range.csv", index=False)
trades.to_csv("backtest_trades.csv", index=False)
try:
    from google.colab import files
    files.download("opening_range.csv")
    files.download("backtest_trades.csv")
    print("Downloads started. Check your browser's downloads.")
except ImportError:
    print("Saved opening_range.csv and backtest_trades.csv")